In [ ]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import confusion_matrix

In [ ]:
# Gabriel Marques
# Email: gmarques@crimson.ua.edu
# CS 451

# Task 1
# Import the MNIST dataset and classify handwritten digits using a Multilayer Perceptron (MLP)

EPOCHS = 5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Normalize pixels to [0,1] then flatten each image 28x28 into 484 dimensional vector
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])

# Load data
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=False)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

# Define MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256,10)
        )
    
    def forward(self, x):
        return self.model(x)
        
model = MLP().to(device)
print(model)

# Loss function + optimizder
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train the model for several epochs
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        model_output = model(X)
        loss = criterion(model_output, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        preds = model_output.argmax(dim=1)
        correct += (preds==y).sum().item()
        total += y.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # Evaluate on test set
    model.eval()
    test_correct = 0
    test_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(device)
            y = y.to(device)

            model_output = model(X)
            preds = model_output.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)
            all_preds.append(preds)
            all_labels.append(y)

    test_acc = test_correct / test_total
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    cm = confusion_matrix(all_labels, all_preds)

    print(f'Epoch {epoch}/{EPOCHS} | train loss: {train_loss:.4f} | train accuracy: {train_acc:.4f} | test accuracy: {test_acc:.4f}')
    print('Confusion Matrix:\n', cm)

In [ ]:
# Task 2
# Import the MNIST dataset and classify handwritten digits using a Convolutional Neural Network (CNN).

EPOCHS = 5
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Normalize pixel values to [0,1]
transform = transforms.ToTensor()

# Load data
train_ds = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=False)
test_ds = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

# Define CNN
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,10)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = CNN().to(device)
print(model)

# Loss function + optimizder
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train the model
for epoch in range(1, EPOCHS+1):
    model.train()
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        model_output = model(X)
        loss = criterion(model_output, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X.size(0)
        preds = model_output.argmax(dim=1)
        correct += (preds==y).sum().item()
        total += y.size(0)

    train_loss = total_loss / total
    train_acc = correct / total

    # Evaluate on test set
    model.eval()
    test_correct = 0
    test_total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X, y in test_loader:
            X = X.to(device)
            y = y.to(device)

            model_output = model(X)
            preds = model_output.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)
            all_preds.append(preds)
            all_labels.append(y)

    test_acc = test_correct / test_total
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    cm = confusion_matrix(all_labels, all_preds)

    print(f'Epoch {epoch}/{EPOCHS} | train loss: {train_loss:.4f} | train accuracy: {train_acc:.4f} | test accuracy: {test_acc:.4f}')
    print('Confusion Matrix:\n', cm)

# Compare against MLP, Which performs better and why
print('CNN usually performs better than MLP on this dataset because it learns spatial features using filters.')
print('On the other hand, MLP flattens the image which loses spatial structure, so it needs more parameterss to learn the same patterns.')